# Manager DNA: Active ETF Behavioral Analysis

**Project Goal:** Reverse-engineer the behavioral DNA of active ETF managers using:
1. **Fama-French 5-Factor** rolling regressions (dimensionality reduction)
2. **Gaussian Mixture Model** regime classification (market state detection)
3. **PCA + Bipartite Network** clustering (managerial archetype mapping)

**Math & CS Foundations:**
- OLS regression, Rolling windows, Active return decomposition
- GMM: $p(X_t) = \sum_{k=1}^{K} \pi_k \mathcal{N}(X_t \mid \mu_k, \Sigma_k)$
- PCA on covariance matrix of factor loadings
- Bipartite graph $G = (U, V, E)$ where $U$ = Funds, $V$ = Super-Styles

---

## 0. Setup
Clone the repo and install dependencies. Skip if running locally.

In [ ]:
# -- Colab setup (skip if local) --
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/james-kidd/manager-dna.git
    %cd manager-dna
    !pip install -e ".[notebook]" -q
else:
    # Local: just make sure you've run `pip install -e .` from the project root
    pass

import sys
sys.path.insert(0, 'src')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

from manager_dna import ManagerialFactorExtractor, MarketRegimeModel, ManagerialDNAMapper
from manager_dna.factor_extraction import FF_FACTORS

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load config (edit config.yaml to change parameters)
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

os.makedirs(cfg['output']['dir'], exist_ok=True)
print('Config loaded. Fund universe:', cfg['fund_universe'])

## Stage 1: Fama-French Active Factor Extraction

For each fund, we compute:
$$\Delta R_{f,t} = \alpha_f + \beta_1(\text{MKT-RF}) + \beta_2(\text{SMB}) + \beta_3(\text{HML}) + \beta_4(\text{RMW}) + \beta_5(\text{CMA}) + \epsilon$$

using a **rolling OLS** window (default 63 trading days = 1 quarter).

In [ ]:
fcfg = cfg['factor_extraction']
fund_universe = cfg['fund_universe']

all_loadings = {}
for ticker in fund_universe:
    print(f'\n--- {ticker} ---')
    ext = ManagerialFactorExtractor(
        fund_ticker=ticker,
        benchmark_ticker=fcfg['benchmark_ticker'],
        start_date=fcfg['start_date'],
        window=fcfg['rolling_window'],
    )
    try:
        ext.fetch_data()
        loadings = ext.extract_rolling_factors()
        all_loadings[ticker] = loadings
    except Exception as e:
        print(f'SKIP {ticker}: {e}')

print(f'\nSuccessfully processed {len(all_loadings)}/{len(fund_universe)} funds.')

In [ ]:
# Visualize rolling betas for the first fund
first_ticker = list(all_loadings.keys())[0]
ax = all_loadings[first_ticker][FF_FACTORS].plot(title=f'{first_ticker} Rolling FF Betas')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('Active Beta Loading')
plt.tight_layout()
plt.show()

## Stage 2: GMM Regime Modeling

Fit a Gaussian Mixture Model over macro features to classify market regimes:
$$p(X_t) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(X_t \mid \mu_k, \Sigma_k)$$

Features: SPY return, VIX level, HY credit spread, 10Y yield change.

In [ ]:
rcfg = cfg['regime_model']
regime = MarketRegimeModel(
    start_date=rcfg['start_date'],
    n_regimes=rcfg['n_regimes'],
)
regime.fetch_macro_data()
regime_data = regime.fit_predict_regimes()

print('\nRegime characteristics:')
regime.get_regime_summary()

In [ ]:
regime.plot_regimes(save_path=os.path.join(cfg['output']['dir'], cfg['output']['regime_plot']))

## Combine: Aggregate Betas by Regime

For each (fund, regime) pair, compute the **mean active beta** across all trading days in that regime. This produces the input matrix for PCA.

In [ ]:
regime_labels = regime_data['Regime']
rows = []

for ticker, loadings in all_loadings.items():
    common = loadings.index.intersection(regime_labels.index)
    if len(common) == 0:
        continue
    merged = loadings.loc[common].copy()
    merged['Regime'] = regime_labels.loc[common]
    for r in range(rcfg['n_regimes']):
        sl = merged[merged['Regime'] == r]
        if len(sl) > 0:
            avg = sl[FF_FACTORS].mean()
            avg.name = f'{ticker}_R{r}'
            rows.append(avg)

df_agg = pd.DataFrame(rows)
print(f'Aggregated matrix: {df_agg.shape}')
df_agg

## Stage 3: PCA Super-Styles & Bipartite Network

Extract orthogonal "Super-Styles" via PCA, then build a bipartite graph:
$$G = (U, V, E) \quad \text{where } U = \text{Funds}, \; V = \text{Super-Styles (PCs)}, \; E = \text{PCA loading}$$

In [ ]:
dcfg = cfg['dna_mapper']
n_comp = min(dcfg['n_components'], len(df_agg))

mapper = ManagerialDNAMapper(n_components=n_comp)
fund_pca, super_styles = mapper.extract_super_styles(df_agg)

print('\nSuper-Style compositions (what each PC loads on):')
display(super_styles)

print('\nFund-regime DNA loadings:')
display(fund_pca)

In [ ]:
network = mapper.build_bipartite_network(edge_threshold=dcfg['edge_threshold'])
mapper.visualize_network(
    network,
    save_path=os.path.join(cfg['output']['dir'], cfg['output']['network_plot'])
)

## Results Interpretation

- **Blue edges** = fund loads *positively* on that Super-Style (long tilt)
- **Red dashed edges** = fund loads *negatively* (short/underweight tilt)
- **Edge thickness** = conviction strength (absolute PCA loading)
- **Funds clustering to the same PC** across regimes = consistent style
- **Funds shifting between PCs** across regimes = tactical/drifting manager

### Next Steps
- [ ] Add Hull delta-equivalent adjustment for options-holding funds
- [ ] Implement style drift detection (graph topology change between regimes)
- [ ] Add more funds to `config.yaml` → `fund_universe`
- [ ] Export results to interactive Plotly/D3 visualization